# nb14 — HGS Algorithm Improvements

**Data:** 2026-03-26

## Diagnóstico

Problema encontrado na análise dos resultados nb13:
- Elliptic++ k=5%: `partition_coverage=0.873` mas `fraud_coverage=0.434`
- **16.219 fraud edges estão na partição certa mas são CORTADAS pelo budget cap**
- Causa raiz: fraud edges dentro de uma comunidade têm score MENOR do que edges de ruído → top-B por score remove preferencialmente as fraudes

## Melhorias Testadas

1. **HGS_v2_NC** — Budget cap por cobertura máxima de nós (greedy) em vez de top-B por score
2. **HGS_v2_SW** — Leiden com pesos de score no grafo Lk  
3. **HGS_v2_BOTH** — As duas melhorias combinadas

## Datasets
- Elliptic (baseline, k=1%,5%)
- Bitcoin-OTC (δ alto, k=1%,5%)
- Yelp Fraud (k=1%) — se disponível localmente

In [ ]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import igraph as ig
import leidenalg
import math
import time
import os
import sys
from collections import defaultdict

BASE = os.path.dirname(os.path.abspath('.'))
DATA_DIR = os.path.join(BASE, 'data')
RESULTS_DIR = os.path.join(BASE, 'results', 'nb14_improvements')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Imports OK')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
# Cell 2 — Funções utilitárias de avaliação (BCCS-P v4 metrics)

def evaluate_bccs_p(cases, src, dst, y, budget_B=100):
    """
    Avalia casos segundo métricas BCCS-P v4.
    
    fraud_coverage = |{e ∈ F : e ∈ ∪P(Cᵢ)}| / |F|
    partition_coverage = |{e ∈ F : endpoints no mesmo caso}| / |F|
    """
    y = np.asarray(y, dtype=int)
    total_fraud = int(y.sum())
    if total_fraud == 0:
        return {}
    
    # Fraud coverage: fraud edges no presented set
    all_presented = set()
    all_induced = set()
    for c in cases:
        all_presented.update(c.get('presented_edges', c.get('induced_edges', []))[:budget_B])
        all_induced.update(c.get('induced_edges', []))
    
    presented_arr = np.array(list(all_presented), dtype=np.int64)
    induced_arr = np.array(list(all_induced), dtype=np.int64)
    
    valid_pres = presented_arr[presented_arr < len(y)]
    valid_ind = induced_arr[induced_arr < len(y)]
    
    fraud_cov = float(y[valid_pres].sum()) / total_fraud if len(valid_pres) > 0 else 0.0
    part_cov = float(y[valid_ind].sum()) / total_fraud if len(valid_ind) > 0 else 0.0
    
    # Yield@100: purity do caso top-ranked
    cases_sorted = sorted(cases, key=lambda c: -c.get('score', 0))
    if cases_sorted:
        top = cases_sorted[0]
        pres = top.get('presented_edges', top.get('induced_edges', []))
        valid_top = np.array([i for i in pres if i < len(y)], dtype=np.int64)
        yield_b = float(y[valid_top].sum()) / max(len(valid_top), 1) if len(valid_top) > 0 else 0.0
    else:
        yield_b = 0.0
    
    # AUC purity
    purities = []
    for c in cases_sorted:
        pres = c.get('presented_edges', c.get('induced_edges', []))
        valid = np.array([i for i in pres if i < len(y)], dtype=np.int64)
        if len(valid) > 0:
            purities.append(float(y[valid].sum()) / len(valid))
    
    if purities:
        cumsum = np.cumsum(purities)
        auc = float(np.mean(cumsum / np.arange(1, len(purities)+1)))
    else:
        auc = 0.0
    
    ind_sizes = np.array([len(c.get('induced_edges', [])) for c in cases])
    
    return {
        'n_cases': len(cases),
        'fraud_coverage': fraud_cov,
        'partition_coverage': part_cov,
        'yield_b100': yield_b,
        'auc_purity': auc,
        'edges_max': float(ind_sizes.max()) if len(ind_sizes) > 0 else 0,
        'edges_median': float(np.median(ind_sizes)) if len(ind_sizes) > 0 else 0,
    }

print('Funções de avaliação OK')

In [ ]:
# Cell 3 — HGS baseline (versão original do nb04_core_cells.py)

def build_Lk(src_sel, dst_sel, ts_sel, delta_L, max_hub_edges=500):
    node_map = defaultdict(list)
    for i in range(len(src_sel)):
        node_map[int(src_sel[i])].append((i, int(ts_sel[i])))
        node_map[int(dst_sel[i])].append((i, int(ts_sel[i])))
    lk_edges = []
    for node, elist in node_map.items():
        if len(elist) < 2:
            continue
        elist.sort(key=lambda x: x[1])
        if len(elist) > max_hub_edges:
            elist = elist[:max_hub_edges]
        for i in range(len(elist)):
            ei, ti = elist[i]
            for j in range(i+1, len(elist)):
                ej, tj = elist[j]
                if tj - ti > delta_L:
                    break
                if ei != ej:
                    lk_edges.append((min(ei, ej), max(ei, ej)))
    # deduplicate
    return list(set(lk_edges))


def build_Lk_weighted(src_sel, dst_sel, ts_sel, scores_sel, delta_L, max_hub_edges=500):
    """Versão com pesos: peso = avg(score_ei, score_ej)"""
    node_map = defaultdict(list)
    for i in range(len(src_sel)):
        node_map[int(src_sel[i])].append((i, int(ts_sel[i])))
        node_map[int(dst_sel[i])].append((i, int(ts_sel[i])))
    edge_weight = {}  # (min,max) -> max weight visto
    for node, elist in node_map.items():
        if len(elist) < 2:
            continue
        elist.sort(key=lambda x: x[1])
        if len(elist) > max_hub_edges:
            elist = elist[:max_hub_edges]
        for i in range(len(elist)):
            ei, ti = elist[i]
            for j in range(i+1, len(elist)):
                ej, tj = elist[j]
                if tj - ti > delta_L:
                    break
                if ei != ej:
                    key = (min(ei, ej), max(ei, ej))
                    w = float(scores_sel[ei] + scores_sel[ej]) / 2.0
                    if key not in edge_weight or w > edge_weight[key]:
                        edge_weight[key] = w
    edges = list(edge_weight.keys())
    weights = [edge_weight[e] for e in edges]
    return edges, weights


def cap_by_score(induced_edges, scores, budget_B):
    """Original: top-B por score."""
    if len(induced_edges) <= budget_B:
        return list(induced_edges)
    idx_arr = np.array(induced_edges, dtype=np.int64)
    valid = idx_arr[idx_arr < len(scores)]
    sc = scores[valid]
    top_b = valid[np.argsort(-sc)[:budget_B]]
    return top_b.tolist()


def cap_by_node_coverage(induced_edges, scores, src, dst, budget_B):
    """
    HGS_v2_NC: Greedy max-node-coverage.
    
    Fase 1: spanning tree ganancioso (max score de cada nó)
      - Para cada nó não coberto, adiciona a edge de maior score que o cobre
    Fase 2: preenche budget restante com top por score
    
    Garante que todo nó do caso tem ao menos 1 edge apresentada.
    Isso evita que fraud edges de score baixo sejam excluídas.
    """
    if len(induced_edges) <= budget_B:
        return list(induced_edges)
    
    idx_arr = np.array(induced_edges, dtype=np.int64)
    valid = idx_arr[idx_arr < len(scores)]
    if len(valid) == 0:
        return []
    
    sc = scores[valid]
    # sort by score descending for efficiency
    order = np.argsort(-sc)
    sorted_edges = valid[order]
    sorted_scores = sc[order]
    
    # Fase 1: greedy node coverage
    covered = set()
    selected = []
    selected_set = set()
    remaining_mask = np.ones(len(sorted_edges), dtype=bool)
    
    # Primeiro pass: para cada nó, garantir cobertura com a melhor edge disponível
    # Nós únicos neste caso
    edge_src = src[sorted_edges] if sorted_edges.max() < len(src) else np.array([])
    edge_dst = dst[sorted_edges] if sorted_edges.max() < len(dst) else np.array([])
    
    if len(edge_src) == 0 or len(edge_dst) == 0:
        return cap_by_score(induced_edges, scores, budget_B)
    
    all_nodes = set(edge_src.tolist()) | set(edge_dst.tolist())
    
    # Greedy: iterar edges em ordem de score (já ordenadas)
    # Adicionar edge se cobre ao menos 1 nó novo, até cobrir todos ou atingir budget
    for i in range(len(sorted_edges)):
        if len(selected) >= budget_B:
            break
        eid = int(sorted_edges[i])
        s_node = int(edge_src[i])
        d_node = int(edge_dst[i])
        # Adiciona se cobre nó novo OU se ainda há budget e todos cobertos
        if s_node not in covered or d_node not in covered:
            selected.append(eid)
            selected_set.add(eid)
            covered.add(s_node)
            covered.add(d_node)
    
    # Fase 2: preencher budget restante com top-score não selecionados
    for i in range(len(sorted_edges)):
        if len(selected) >= budget_B:
            break
        eid = int(sorted_edges[i])
        if eid not in selected_set:
            selected.append(eid)
            selected_set.add(eid)
    
    return selected


print('Funções de cap OK')

In [ ]:
# Cell 4 — HGS genérico com variantes

def hgs_generic(
    scores, src, dst, timestamps, y,
    k=0.05, delta_L=7, resolution=1.0, budget_B=100, seed=42,
    variant='baseline'  # 'baseline', 'nc', 'sw', 'both'
):
    """
    HGS (Hierarchical Graph Segmentation) com variantes de melhoria.
    
    variant:
      'baseline' — HGS original (top-B por score, Leiden sem peso)
      'nc'       — Node-Coverage cap (budget cap greedy por cobertura de nós)
      'sw'       — Score-Weighted Leiden (pesos no grafo Lk)
      'both'     — NC + SW combinados
    """
    use_nc = variant in ('nc', 'both')
    use_sw = variant in ('sw', 'both')
    
    scores = np.asarray(scores, dtype=float)
    src = np.asarray(src, dtype=np.int64)
    dst = np.asarray(dst, dtype=np.int64)
    ts = np.asarray(timestamps, dtype=np.int64)
    N = len(scores)
    K = max(1, int(math.ceil(k * N)))
    sel = np.argsort(-scores)[:K]
    
    src_sel = src[sel]
    dst_sel = dst[sel]
    ts_sel = ts[sel]
    sc_sel = scores[sel]
    max_node = int(max(src.max(), dst.max())) + 1
    
    print(f'  [{variant}] K={K:,} top edges (of {N:,})')
    
    # --- Step 1: WCC ---
    all_nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap = {int(n): i for i, n in enumerate(all_nodes)}
    edges_compact = [(nmap[int(s)], nmap[int(d)]) for s, d in zip(src_sel, dst_sel)]
    g_node = ig.Graph(n=len(all_nodes), edges=edges_compact, directed=False)
    g_node.simplify()
    wcc = g_node.connected_components(mode='weak')
    wcc_mem = np.array(wcc.membership, dtype=np.int64)
    n_wcc = int(wcc_mem.max()) + 1
    edge_wcc = np.array([wcc_mem[nmap[int(s)]] for s in src_sel], dtype=np.int64)
    
    # Agrupar edges e nós por WCC
    wcc_edge_lists = [[] for _ in range(n_wcc)]
    wcc_node_sets = [set() for _ in range(n_wcc)]
    for i in range(K):
        w = int(edge_wcc[i])
        wcc_edge_lists[w].append(i)
        wcc_node_sets[w].update([int(src_sel[i]), int(dst_sel[i])])
    
    # Contar induced edges por WCC
    wcc_gid = -np.ones(max_node, dtype=np.int64)
    for w, nodes in enumerate(wcc_node_sets):
        for nid in nodes:
            wcc_gid[nid] = w
    g_src_w = np.where(src < max_node, wcc_gid[src], -1)
    g_dst_w = np.where(dst < max_node, wcc_gid[dst], -1)
    mask_w = (g_src_w == g_dst_w) & (g_src_w >= 0)
    idx_w = np.where(mask_w)[0]
    wcc_ind_count = np.zeros(n_wcc, dtype=np.int64)
    if len(idx_w) > 0:
        gw = g_src_w[idx_w]
        uq, cnt = np.unique(gw, return_counts=True)
        wcc_ind_count[uq] = cnt
    
    n_small = int((wcc_ind_count[wcc_ind_count > 0] <= budget_B).sum())
    n_large = int((wcc_ind_count > budget_B).sum())
    print(f'  WCC: {n_wcc:,} | {n_small} small, {n_large} large (need Leiden)')
    
    # --- Step 2: Leiden hierárquico para WCCs grandes ---
    final_mem = np.full(K, -1, dtype=np.int64)
    next_id = 0
    leiden_calls = 0
    
    for w in range(n_wcc):
        comp = wcc_edge_lists[w]
        if not comp:
            continue
        need_split = (budget_B is not None and wcc_ind_count[w] > budget_B and len(comp) >= 2)
        if not need_split:
            for i in comp:
                final_mem[i] = next_id
            next_id += 1
        else:
            comp_arr = np.array(comp, dtype=np.int64)
            
            if use_sw:
                # Score-Weighted Leiden
                lk_edges, lk_weights = build_Lk_weighted(
                    src_sel[comp_arr], dst_sel[comp_arr],
                    ts_sel[comp_arr], sc_sel[comp_arr], delta_L
                )
            else:
                # Original: sem peso
                lk_edges = build_Lk(
                    src_sel[comp_arr], dst_sel[comp_arr],
                    ts_sel[comp_arr], delta_L
                )
                lk_weights = None
            
            if not lk_edges:
                for i in comp:
                    final_mem[i] = next_id
                    next_id += 1
            else:
                g_local = ig.Graph(n=len(comp), edges=lk_edges, directed=False)
                g_local.simplify()
                
                if use_sw and lk_weights is not None:
                    # Normalizar pesos para [0,1]
                    w_arr = np.array(lk_weights)
                    w_min, w_max = w_arr.min(), w_arr.max()
                    if w_max > w_min:
                        w_norm = (w_arr - w_min) / (w_max - w_min)
                    else:
                        w_norm = np.ones_like(w_arr)
                    g_local.es['weight'] = w_norm.tolist()
                    part = leidenalg.find_partition(
                        g_local, leidenalg.RBConfigurationVertexPartition,
                        weights='weight',
                        resolution_parameter=resolution, seed=seed
                    )
                else:
                    part = leidenalg.find_partition(
                        g_local, leidenalg.RBConfigurationVertexPartition,
                        resolution_parameter=resolution, seed=seed
                    )
                
                leiden_calls += 1
                local_mem = np.array(part.membership, dtype=np.int64)
                n_sub = int(local_mem.max()) + 1
                for j, idx in enumerate(comp):
                    final_mem[idx] = next_id + int(local_mem[j])
                next_id += n_sub
    
    print(f'  Leiden calls: {leiden_calls} | Total partitions: {next_id}')
    
    # --- Step 3: Montar casos (voto majoritário) ---
    n_total = next_id
    node_votes = defaultdict(lambda: defaultdict(int))
    for i in range(K):
        g = int(final_mem[i])
        if g < 0:
            continue
        node_votes[int(src_sel[i])][g] += 1
        node_votes[int(dst_sel[i])][g] += 1
    
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': [], 'presented_edges': [], 'score': 0.0}
             for _ in range(n_total)]
    for nid, votes in node_votes.items():
        best = max(votes, key=votes.get)
        cases[best]['nodes'].add(nid)
    for i in range(K):
        g = int(final_mem[i])
        if g >= 0:
            cases[g]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]
    
    # --- Step 4: Arestas induzidas ---
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g, case in enumerate(cases):
        for nid in case['nodes']:
            gid_of[nid] = g
    g_src_f = np.where(src < max_node, gid_of[src], -1)
    g_dst_f = np.where(dst < max_node, gid_of[dst], -1)
    mask_f = (g_src_f == g_dst_f) & (g_src_f >= 0)
    idx_f = np.where(mask_f)[0]
    if len(idx_f) > 0:
        gf = g_src_f[idx_f]
        order = np.argsort(gf, kind='stable')
        g_sorted = gf[order]
        i_sorted = idx_f[order]
        uq, cnt = np.unique(g_sorted, return_counts=True)
        for g_id, grp in zip(uq, np.split(i_sorted, np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()
    
    # Case score = max score das seed_edges
    for case in cases:
        if case['seed_edges']:
            valid_seeds = [i for i in case['seed_edges'] if i < len(scores)]
            case['score'] = float(scores[valid_seeds].max()) if valid_seeds else 0.0
    
    # --- Step 5: Budget cap ---
    n_capped = 0
    for case in cases:
        ie = case['induced_edges']
        if len(ie) > budget_B:
            n_capped += 1
            if use_nc:
                case['presented_edges'] = cap_by_node_coverage(ie, scores, src, dst, budget_B)
            else:
                case['presented_edges'] = cap_by_score(ie, scores, budget_B)
        else:
            case['presented_edges'] = list(ie)
    
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    print(f'  Final: {len(cases)} cases | capped={n_capped} | median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
    return cases

print('hgs_generic com variantes OK')

In [ ]:
# Cell 5 — Loader Elliptic

def load_elliptic(data_dir):
    """
    Carrega Elliptic Bitcoin Dataset.
    Score = fração de features de classe 1 (heurístico por timestep).
    """
    edge_path = os.path.join(data_dir, 'elliptic', 'elliptic_bitcoin_dataset', 'elliptic_txs_edgelist.csv')
    class_path = os.path.join(data_dir, 'elliptic', 'elliptic_bitcoin_dataset', 'elliptic_txs_classes.csv')
    feat_path = os.path.join(data_dir, 'elliptic', 'elliptic_bitcoin_dataset', 'elliptic_txs_features.csv')
    
    edges_df = pd.read_csv(edge_path)
    classes_df = pd.read_csv(class_path)
    feats_df = pd.read_csv(feat_path, header=None)
    
    # Renomear colunas
    edges_df.columns = ['src', 'dst']
    classes_df.columns = ['txId', 'class']
    feats_df.columns = ['txId', 'timestep'] + [f'f{i}' for i in range(feats_df.shape[1]-2)]
    
    # Map txId -> index
    all_txs = pd.concat([edges_df['src'], edges_df['dst']]).unique()
    tx_map = {tx: i for i, tx in enumerate(all_txs)}
    
    src_arr = np.array([tx_map[t] for t in edges_df['src']], dtype=np.int64)
    dst_arr = np.array([tx_map[t] for t in edges_df['dst']], dtype=np.int64)
    N = len(src_arr)
    
    # Labels
    class_map = dict(zip(classes_df['txId'], classes_df['class']))
    src_labels = np.array([1 if class_map.get(t, 'unknown') == '1' else 0 for t in edges_df['src']])
    dst_labels = np.array([1 if class_map.get(t, 'unknown') == '1' else 0 for t in edges_df['dst']])
    y = ((src_labels == 1) & (dst_labels == 1)).astype(int)
    
    # Timestamps via timestep do feat
    ts_map = dict(zip(feats_df['txId'], feats_df['timestep']))
    src_ts = np.array([ts_map.get(t, 0) for t in edges_df['src']], dtype=np.int64)
    dst_ts = np.array([ts_map.get(t, 0) for t in edges_df['dst']], dtype=np.int64)
    ts_arr = (src_ts + dst_ts) // 2
    
    # Score = fraud_rate no timestep (heurístico, como nb04)
    # Usar feature f0 como proxy de score (primeira feature contextual)
    feat_map = dict(zip(feats_df['txId'], feats_df['f0']))
    src_feat = np.array([feat_map.get(t, 0.0) for t in edges_df['src']], dtype=float)
    dst_feat = np.array([feat_map.get(t, 0.0) for t in edges_df['dst']], dtype=float)
    scores = (src_feat + dst_feat) / 2.0
    # Normalizar
    s_min, s_max = scores.min(), scores.max()
    if s_max > s_min:
        scores = (scores - s_min) / (s_max - s_min)
    
    print(f'Elliptic: {N:,} edges | fraud={y.sum():,} ({100*y.mean():.2f}%)')
    return scores, src_arr, dst_arr, ts_arr, y


def load_bitcoin_otc(data_dir):
    """Carrega Bitcoin-OTC. Score = abs(rating)/10, y = rating < 0."""
    path = os.path.join(data_dir, 'bitcoin_otc', 'soc-sign-bitcoinotc.csv')
    df = pd.read_csv(path, header=None, names=['src', 'dst', 'rating', 'ts'])
    df = df.dropna()
    
    all_nodes = pd.concat([df['src'], df['dst']]).unique()
    nmap = {n: i for i, n in enumerate(all_nodes)}
    
    src_arr = np.array([nmap[n] for n in df['src']], dtype=np.int64)
    dst_arr = np.array([nmap[n] for n in df['dst']], dtype=np.int64)
    ts_arr = np.array(df['ts'], dtype=np.int64)
    ratings = np.array(df['rating'], dtype=float)
    
    # Fraude = rating negativo
    y = (ratings < 0).astype(int)
    # Score = magnitude do rating negativo (mais negativo = mais suspeito)
    scores = np.where(ratings < 0, -ratings / 10.0, 0.0)
    # Adicionar ruído para diferenciar edges
    scores = scores + np.random.RandomState(42).uniform(0, 0.01, len(scores))
    scores = np.clip(scores, 0, 1)
    
    print(f'Bitcoin-OTC: {len(src_arr):,} edges | fraud={y.sum():,} ({100*y.mean():.2f}%)')
    return scores, src_arr, dst_arr, ts_arr, y

print('Loaders OK')

In [ ]:
# Cell 6 — Carregar datasets

print('=== Carregando Elliptic ===')
ell_scores, ell_src, ell_dst, ell_ts, ell_y = load_elliptic(DATA_DIR)

print('\n=== Carregando Bitcoin-OTC ===')
btc_scores, btc_src, btc_dst, btc_ts, btc_y = load_bitcoin_otc(DATA_DIR)

In [ ]:
# Cell 7 — Experimentos: Elliptic

variants = ['baseline', 'nc', 'sw', 'both']
k_values = [0.01, 0.05]
results_ell = []

for k in k_values:
    for v in variants:
        print(f'\n--- Elliptic k={k*100:.0f}% variant={v} ---')
        t0 = time.perf_counter()
        cases = hgs_generic(
            ell_scores, ell_src, ell_dst, ell_ts, ell_y,
            k=k, delta_L=7, resolution=1.0, budget_B=100, seed=42,
            variant=v
        )
        t1 = time.perf_counter()
        metrics = evaluate_bccs_p(cases, ell_src, ell_dst, ell_y, budget_B=100)
        metrics['dataset'] = 'Elliptic'
        metrics['method'] = f'HGS_v2_{v.upper()}' if v != 'baseline' else 'HGS_baseline'
        metrics['k_pct'] = k * 100
        metrics['time_s'] = round(t1 - t0, 4)
        results_ell.append(metrics)
        print(f'  fraud_cov={metrics["fraud_coverage"]:.4f} | part_cov={metrics["partition_coverage"]:.4f} | yield@100={metrics["yield_b100"]:.4f} | t={t1-t0:.2f}s')

df_ell = pd.DataFrame(results_ell)
print('\n=== RESULTADOS ELLIPTIC ===')
print(df_ell[['dataset','method','k_pct','fraud_coverage','partition_coverage','yield_b100','auc_purity','time_s']].to_string())

In [ ]:
# Cell 8 — Experimentos: Bitcoin-OTC

results_btc = []

for k in k_values:
    for v in variants:
        print(f'\n--- Bitcoin-OTC k={k*100:.0f}% variant={v} ---')
        t0 = time.perf_counter()
        cases = hgs_generic(
            btc_scores, btc_src, btc_dst, btc_ts, btc_y,
            k=k, delta_L=7, resolution=1.0, budget_B=100, seed=42,
            variant=v
        )
        t1 = time.perf_counter()
        metrics = evaluate_bccs_p(cases, btc_src, btc_dst, btc_y, budget_B=100)
        metrics['dataset'] = 'Bitcoin-OTC'
        metrics['method'] = f'HGS_v2_{v.upper()}' if v != 'baseline' else 'HGS_baseline'
        metrics['k_pct'] = k * 100
        metrics['time_s'] = round(t1 - t0, 4)
        results_btc.append(metrics)
        print(f'  fraud_cov={metrics["fraud_coverage"]:.4f} | part_cov={metrics["partition_coverage"]:.4f} | yield@100={metrics["yield_b100"]:.4f} | t={t1-t0:.2f}s')

df_btc = pd.DataFrame(results_btc)
print('\n=== RESULTADOS BITCOIN-OTC ===')
print(df_btc[['dataset','method','k_pct','fraud_coverage','partition_coverage','yield_b100','auc_purity','time_s']].to_string())

In [ ]:
# Cell 9 — Combinar resultados e salvar

df_all = pd.concat([df_ell, df_btc], ignore_index=True)

# Reorganizar colunas
cols = ['dataset','method','k_pct','fraud_coverage','partition_coverage',
        'yield_b100','auc_purity','n_cases','edges_max','edges_median','time_s']
df_all = df_all[cols]

out_path = os.path.join(RESULTS_DIR, 'nb14_improvements_results.csv')
df_all.to_csv(out_path, index=False)
print(f'Salvo em: {out_path}')

# Resumo: ganho do NC vs baseline
print('\n=== GANHO NC vs BASELINE (fraud_coverage) ===')
for ds in df_all['dataset'].unique():
    for k in k_values:
        sub = df_all[(df_all['dataset']==ds) & (df_all['k_pct']==k*100)]
        base = sub[sub['method']=='HGS_baseline']['fraud_coverage'].values
        nc = sub[sub['method']=='HGS_v2_NC']['fraud_coverage'].values
        both = sub[sub['method']=='HGS_v2_BOTH']['fraud_coverage'].values
        if len(base) > 0 and len(nc) > 0:
            delta_nc = float(nc[0]) - float(base[0])
            delta_both = float(both[0]) - float(base[0]) if len(both) > 0 else 0
            print(f'  {ds} k={k*100:.0f}%: NC Δ={delta_nc:+.4f} | BOTH Δ={delta_both:+.4f}')

print('\n=== TABELA COMPLETA ===')
print(df_all.to_string(index=False))

In [ ]:
# Cell 10 — Diagnóstico: por que NC ajuda?
# Analisa a distribuição de scores dentro das presented_edges para fraud vs não-fraud

print('=== Diagnóstico: distribuição de scores por tipo de edge ===')
print('Dataset: Elliptic k=5%')

K_diag = max(1, int(math.ceil(0.05 * len(ell_scores))))
sel_diag = np.argsort(-ell_scores)[:K_diag]

# Todas as top-K edges
y_topk = ell_y[sel_diag]
sc_topk = ell_scores[sel_diag]

print(f'  Top-K edges: {K_diag:,} | fraud={y_topk.sum():,} ({100*y_topk.mean():.2f}%)')
print(f'  Score médio (fraud): {sc_topk[y_topk==1].mean():.4f}')
print(f'  Score médio (non-fraud): {sc_topk[y_topk==0].mean():.4f}')

# Induced edges que NÃO são seed (são adicionadas porque conectam nós do mesmo caso)
# Para detectar isso, pegar todos os induced edges dos casos do HGS baseline
print('\n  Rodando HGS baseline k=5% para análise...')
cases_diag = hgs_generic(
    ell_scores, ell_src, ell_dst, ell_ts, ell_y,
    k=0.05, delta_L=7, resolution=1.0, budget_B=100, seed=42,
    variant='baseline'
)

# Separar induced edges em: seed (top-K) vs induced-only
top_k_set = set(sel_diag.tolist())
all_induced = []
for c in cases_diag:
    all_induced.extend(c['induced_edges'])

induced_arr = np.array(all_induced, dtype=np.int64)
valid_ind = induced_arr[induced_arr < len(ell_y)]

is_seed = np.array([i in top_k_set for i in valid_ind])
is_fraud = ell_y[valid_ind]
sc_ind = ell_scores[valid_ind]

print(f'\n  Induced-only edges (não seed): {(~is_seed).sum():,}')
print(f'  Fraud rate nas induced-only: {is_fraud[~is_seed].mean()*100:.2f}%')
print(f'  Fraud rate nas seed: {is_fraud[is_seed].mean()*100:.2f}%')
print(f'  Score médio induced-only (fraud): {sc_ind[~is_seed & (is_fraud==1)].mean():.4f}')
print(f'  Score médio induced-only (non-fraud): {sc_ind[~is_seed & (is_fraud==0)].mean():.4f}')

print('\nConclusion: se fraud_score < nonfraud_score nas induced-only, NC ajuda muito!')